## {PATTERN_NAME} Knowledge Graph Retrieve & Generation Content

This notebook is the Neo4j knowledge-graph counterpart to the MaaS inference notebook. It runs graph-only retrieval and answer generation over the collection created by the knowledge-graph indexing notebook.

### &#x1F4CB; Contents 
This notebook contains the following sections:

- **[Setup](#Setup)**
- **[Prepare MaaS client](#Prepare-MaaS-client)**
- **[Initialize RAG Components](#Initialize-RAG-Components)**
   - [Initialize Foundation Model](#Initialize-Foundation-Model)
   - [Initialize Vector Store Client](#Initialize-Vector-Store-Client)
   - [Initialize Retriever](#Initialize-Retriever)
   - [Query RAG Pattern](#Query-RAG-Pattern)
- **[Next steps](#Next-steps)**
   - [Load Test Data](#Load-Test-Data)
   - [Configure S3 Credentials](#Configure-S3-Credentials)
   - [Initialize S3 Client](#Initialize-S3-Client)
   - [Load Benchmark Data](#Load-Benchmark-Data)
   - [Build Evaluation Data](#Build-Evaluation-Data)
   - [Evaluate Response](#Evaluate-Response)
- **[Summary](#Summary)**

---

## Setup

This section installs all the required Python packages for running the RAG experiment:
- **ai4rag**: The main RAG framework for AutoRAG experiments

In [ ]:
%pip install 'ai4rag~={AI4RAG_VERSION}' | tail -n 1

---

## Prepare MaaS client

OpenShift AI Models-as-a-Service (MaaS) serves every model — foundation and embedding — from a single OpenAI-compatible endpoint. This section builds one client from the MaaS base URL and API key, and reuses it for all models.

**Prerequisites:**
- `MAAS_API_KEY`: Your authentication key for the MaaS API
- `MAAS_BASE_URL`: The complete OpenAI-compatible endpoint URL, used verbatim (e.g. `https://<host>/v1`)

&#x1F4A1; **Tip**: In OpenShift AI Workbench, you can add these as environment variables or data connections to avoid entering them manually each time.

In [ ]:
import getpass
import logging
import os
import warnings

from ai4rag.utils.clients.maas_client import create_maas_client

warnings.filterwarnings("ignore")
logging.getLogger("httpx").propagate = False

MAAS_API_KEY = os.getenv("MAAS_API_KEY") or getpass.getpass("Please enter 'MAAS_API_KEY': ")
MAAS_BASE_URL = os.getenv("MAAS_BASE_URL") or getpass.getpass("Please enter 'MAAS_BASE_URL': ")

client = create_maas_client(base_url=MAAS_BASE_URL, api_key=MAAS_API_KEY)

---

## Initialize RAG Components

This section sets up all the components needed for the RAG pattern: foundation model, vector store, retriever, and the RAG pattern itself.

### Initialize Foundation Model

The foundation model is the core language model that generates responses. This section configures:
- **Model ID**: The specific MaaS model to use for generation
- **System Message**: Instructions that define the model's behavior and role
- **User Message Template**: The format for user queries
- **Context Template**: How retrieved context is incorporated into prompts
- **Language**: The response language detected for this pattern, applied so generated answers match the benchmark's language

These templates control how the RAG system structures prompts to the language model.

In [ ]:
from ai4rag.rag.foundation_models.base_model import Language
from ai4rag.rag.foundation_models.openai_model import OpenAIFoundationModel

chat_model_id = """{FM_MODEL_ID}"""
system_message_text = """{SYSTEM_MESSAGE}"""
user_message_text = """{USER_MESSAGE}"""
context_template_text = """{CONTEXT_TEXT}"""
language = Language(**{LANGUAGE})

foundation_model = OpenAIFoundationModel(
    client=client,
    model_id=chat_model_id,
    system_message_text=system_message_text,
    user_message_text=user_message_text,
    context_template_text=context_template_text,
    language=language,
)

## Connect to the Neo4j graph

`Neo4jGraphStore` uses a collection-specific vector index to select seed chunks. Its local route can discover other individual chunks through shared entities and, when configured, entity-to-entity relationships.

In [ ]:
from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.vector_store.config import Neo4jConfig
from ai4rag.rag.vector_store.neo4j import Neo4jGraphStore

embedding_model = OpenAIEmbeddingModel(
    client=client,
    model_id="{EMBEDDING_MODEL_ID}",
    params=OpenAIEmbeddingParams(**{EMBEDDING_PARAMS}),
)
collection_name = "{COLLECTION_NAME}"
vector_store = Neo4jGraphStore(
    embedding_model=embedding_model,
    config=Neo4jConfig.from_env(),
    collection_name=collection_name,
)

## Configure graph retrieval and generate an answer

The retriever returns at most `number_of_chunks` individual stored chunks after fusing direct vector and local graph rankings. By default, shared-entity expansion is enabled with a limit of five neighbors per seed; entity-to-entity traversal is disabled unless its three controls are positive. This uses the same MaaS foundation model and response flow as the standard inference notebook.

In [ ]:
from ai4rag.rag.retrieval.retriever import Retriever
{RAG_TEMPLATE_IMPORT}

retriever = Retriever(
    vector_store=vector_store,
    method="simple",
    number_of_chunks={NUMBER_OF_CHUNKS},
    search_mode="graph",
    search_kwargs={GRAPH_RETRIEVAL_CONFIG},
)
rag_pattern = {RAG_TEMPLATE_CLASS}(foundation_model=foundation_model, retriever=retriever)

---

### Query RAG Pattern

This section executes the RAG workflow by submitting test questions to the system and generating responses based on retrieved context.

In [ ]:
from pprint import pprint

question = input()
response = rag_pattern.generate(question=question)
pprint(response, indent=4)

## Direct graph-search controls

For inspection outside the RAG pattern, call `vector_store.search()` directly. `entity_neighbor_limit=0` disables shared-entity expansion. Use `entity_pivot_limit`, `entity_relationship_hops`, and `relationship_neighbor_limit` to expand through entity relationships. Each graph-linked chunk is independently rescored; the final `include_scores` values are reciprocal-rank fusion scores. Neo4j accepts only `search_mode="graph"`.

In [ ]:
results = vector_store.search(
    query=question,
    k={NUMBER_OF_CHUNKS},
    search_mode="graph",
    **{GRAPH_RETRIEVAL_CONFIG},
    include_scores=True,
)
for chunk, score in results:
    print(f"score={{score:.4f}} document_id={{chunk.metadata.get('document_id', '')}}")
    print(chunk.text[:500])

In [ ]:
vector_store.close()

---

## Next steps

The following sections provide optional next steps for loading test data, running queries, and evaluating the RAG pattern's performance. These steps are useful for systematic testing and benchmarking, but can be skipped if you prefer to interact with the RAG system directly using the pattern configured above.

---

### Load Test Data

This section prepares the test environment and loads benchmark questions from S3 storage. The test data is used to evaluate the RAG system's performance.

```python
import os
import json
from pathlib import Path
from types import SimpleNamespace

import boto3

logging.getLogger('Test Data Loader component logger').propagate = False
```

### Configure S3 Credentials

To load test data from S3-compatible object storage, you need to provide credentials. If you're using OpenShift AI, these can be configured as data connections.

&#x1F4CC; **Action**: Provide the credentials for your S3 instance if they are not already set in the notebook environment.

&#x1F4A1; **Tip**: In the project, open **Connections** and add an **S3 compatible object storage connection** to a bucket you will use for documents and test data. Open **Workbenches**, edit your workbench, and attach the S3 connection you created so the notebook can read from the bucket. Save and restart the workbench if prompted.

```python
required_vars = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
missing = [var for var in required_vars if not os.environ.get(var)]
if missing:
    raise ValueError(f"Missing required environment variables: {{missing}}")
```

### Initialize S3 Client

Creates an S3 client session using the provided credentials. This client is used to download test data from the specified S3 bucket.

```python
session = boto3.session.Session(
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
s3_client = session.client(
    service_name='s3',
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
)
```

### Load Benchmark Data

Downloads and loads the benchmark test data from S3. The benchmark file should be a JSON file containing:
- **question**: The test question to ask the RAG system
- **correct_answers**: The expected answers for evaluation
- **correct_answer_document_keys**: Keys of the documents that contain the correct information

This data is used to measure the RAG system's accuracy and retrieval quality.

```python
from ai4rag.utils.data.test_data_loader import load_test_data


bucket_name = os.environ['AWS_S3_BUCKET']
key = "{TEST_DATA_KEY}"

test_data_result = load_test_data(
    bucket_name=bucket_name,
    key=key,
    s3_client=s3_client,
)

test_data = test_data_result.data
print(json.dumps(test_data, indent=4, ensure_ascii=False))
```

```python
inference_responses = []

for test_data_item in test_data:
    response = rag_pattern.generate(question=test_data_item["question"])
    inference_responses.append(response)
```

### Build Evaluation Data

This section transforms the RAG system's inference responses into a structured format for evaluation. It combines:
- **Benchmark Data**: The original test questions and expected answers
- **Inference Responses**: The actual responses generated by the RAG system

The resulting evaluation data structure allows for systematic comparison between expected and actual outputs, enabling metric calculation for assessing the RAG system's performance.

```python
from pandas import DataFrame

from ai4rag.core.experiment.utils import build_evaluation_data
from ai4rag.core.experiment.benchmark_data import BenchmarkData


benchmark_data = BenchmarkData(
    DataFrame(
        data=test_data
    )
)

evaluation_data = build_evaluation_data(
    benchmark_data=benchmark_data, 
    inference_response=inference_responses
)
```

### Evaluate Response

This section evaluates the quality of the RAG system's responses by comparing them against the expected answers from the benchmark data. Evaluation metrics may include accuracy, relevance, and retrieval precision.

```python
from ai4rag.evaluator.unitxt_evaluator import UnitxtEvaluator
from ai4rag.evaluator.base_evaluator import MetricType

evaluator = UnitxtEvaluator()
evaluator.evaluate_metrics(evaluation_data=evaluation_data, metrics=(MetricType.ANSWER_CORRECTNESS, MetricType.FAITHFULNESS, MetricType.CONTEXT_CORRECTNESS))
```

---

## Summary

This notebook used MaaS generation with Neo4j graph retrieval. The optional test-data and evaluation sections above can evaluate the resulting graph RAG pattern.